# Carkaroo AI: training both models in Google Colab

This notebook builds the AI of Carkaroo from scratch: it **simulates a factory, makes the dataset, trains the models, tests them honestly and exports them** for the laptop that runs the factory.

Carkaroo has **two** machine-learning models, each with its own job:

| | Model A: the **ranker** | Model B: the **shortage-risk model** |
|---|---|---|
| Question | *Which delivery should the car do first?* | *Will this station run out before a box sent now can arrive?* |
| Output | a score per request; the highest goes first | a probability per station, e.g. **0.92** |
| Learns from | an exact optimizer that tries every possible order (the "teacher") | the future of simulated factory timelines with changing demand |
| Used for | the decision the car receives | the risk % shown per station and the explanation of the urgency |

**Honesty note.** All data in this notebook is **simulated**. We have no real factory history. The shapes of the simulation (shifts, production orders, surges, downtime) are common in real production, but no number comes from a real factory. The ranker is also tested on 14 real factory *layouts* (the KIT AGV data set, CC BY 4.0) in `scripts/evaluate_real_layouts.py`.

**Open it:** [Open in Colab](https://colab.research.google.com/github/noureldinGM/Capstone-AVG/blob/main/notebooks/carkaroo_ai_colab.ipynb). The repository is private, so Colab asks you once to sign in to GitHub and to tick *Include private repos*. Or in Colab: *File → Open notebook → GitHub*.

**How to run (everything in Colab, nothing on your PC):**
1. **Once:** make a read-only GitHub key. GitHub → Settings → Developer settings → Fine-grained personal access tokens → *Generate new token*.
   - Repository access: *Only select repositories* → `Capstone-AVG`.
   - Permissions: *Contents* → **Read-only**. Nothing else.
   - Expiration: 90 days (or the end of the school year).
2. **Once per Google account:** in Colab, click the **key icon** (Secrets) on the left → *Add new secret*.
   - Name: `GITHUB_TOKEN`; value: the token.
   - Switch on *Notebook access*.

   The token stays in your Google account. It is never printed and never saved in the notebook.
3. `Runtime → Run all`. The first cell downloads the project code from GitHub (branch `main`).

**Choose the size** with `MODE` in the first code cell:

| `MODE` | Ranker data | Risk data | Time on free Colab (about) | Use it for |
|---|---|---|---|---|
| `"quick"` | 3,000 situations | 60 timelines | about 5 min | checking that everything works |
| `"standard"` | 12,000 situations + 4,000 in large factories | 120 timelines | about 10 min | a good model in one sitting |
| `"full"` | 30,000 situations + 12,000 in large factories (the same as the project's model) | 180 timelines | about 20–30 min | the final model for the exhibition |

The project's current ranker also had live-operation training (DAgger, section A7, off by default), which can add an hour or more. Keep the browser tab open while it runs: free Colab stops a runtime that is left idle. At the end, the trained models are saved to your Google Drive (`MyDrive/carkaroo_models/`) and downloaded.

*No token?* Set `SOURCE = "zip"` and upload `carkaroo_colab.zip` instead (made on a PC with `python scripts/make_colab_bundle.py`).

## 1. Setup

**What:** install the exact scikit-learn version the laptop uses, download the project code from GitHub, and import it.

**Why the exact version?** A trained model is saved with `joblib`, which stores Python objects as they are. A model saved by one scikit-learn version may not load (or may behave differently) in another. The laptop uses **1.9.1**, so Colab must too.

**Why a token?** The GitHub repository is private, so Colab must prove it may read it. The token is read-only and limited to this one repository, and it is read from Colab's Secrets, never typed into the notebook. Treat it like a password: never paste it in a cell, a chat or a screenshot.

In [ ]:
import io, os, shutil, subprocess, sys, time, urllib.error, urllib.request, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SKLEARN_VERSION = "1.9.1"          # must match the laptop (.venv)
MODE = "quick"                     # "quick" (~5 min) | "standard" (~10 min) | "full" (~20-30 min, like the reports)
SOURCE = "github"                  # "github" = download with GITHUB_TOKEN from Colab Secrets | "zip" = upload carkaroo_colab.zip
REPO, BRANCH = "noureldinGM/Capstone-AVG", "main"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SKLEARN_VERSION}"], check=True)
    WORK = Path("/content")
else:                              # running on a PC (used to test this notebook)
    SOURCE = os.environ.get("CARKAROO_SOURCE", "zip")
    BRANCH = os.environ.get("CARKAROO_BRANCH", BRANCH)
    MODE = os.environ.get("CARKAROO_MODE", MODE)
    WORK = Path(os.environ.get("CARKAROO_WORK", "."))
ROOT = (WORK / "carkaroo").resolve()
if ROOT.exists():
    shutil.rmtree(ROOT)                # start clean when the cell is run again

def github_token():
    if not IN_COLAB:
        return os.environ["GITHUB_TOKEN"]
    from google.colab import userdata
    try:
        return userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise SystemExit("No GITHUB_TOKEN secret: click the key icon on the left, add it and switch on "
                         f"'Notebook access' (see the top of this notebook). ({type(e).__name__})")

if SOURCE == "github":
    req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}",
                                 headers={"Authorization": f"Bearer {github_token()}",
                                          "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req) as r:
            data = r.read()
    except urllib.error.HTTPError as e:
        hint = {401: "the token is wrong or expired", 403: "the token is not allowed to read this repository",
                404: f"the token cannot see {REPO}, or branch '{BRANCH}' does not exist"}.get(e.code, e.reason)
        raise SystemExit(f"Download failed ({e.code}): {hint}.")
    del req                           # forget the request that holds the token
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        top = z.namelist()[0].split("/")[0]
        z.extractall(WORK)
    (WORK / top).rename(ROOT)
    print(f"downloaded {REPO} ({BRANCH}), {len(data) / 1e6:.1f} MB")
else:
    if IN_COLAB:
        from google.colab import files
        print("Choose carkaroo_colab.zip ...")
        bundle = WORK / next(iter(files.upload()))
    else:
        bundle = Path(os.environ["CARKAROO_BUNDLE"])
    with zipfile.ZipFile(bundle) as z:
        z.extractall(WORK)

os.chdir(ROOT)
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
OUT = ROOT / "models_out"
OUT.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
SIZES = {   # ranker situations, random layouts, large-factory situations, test situations, risk timelines, unseen timelines
    "quick":    dict(scen=3000,  layouts=100, plants=0,     test=300,  risk_tl=60,  unseen=20),
    "standard": dict(scen=12000, layouts=300, plants=4000,  test=500,  risk_tl=120, unseen=30),
    "full":     dict(scen=30000, layouts=400, plants=12000, test=1000, risk_tl=180, unseen=40),
}
assert MODE in SIZES, f"MODE must be one of {list(SIZES)}"
SIZE = SIZES[MODE]
print("scikit-learn", sklearn.__version__, "| project at", ROOT, "| MODE =", MODE, SIZE)
assert sklearn.__version__ == SKLEARN_VERSION, "Restart the runtime (Runtime -> Restart) and run again"

## 2. Choosing the machine-learning problem

Before any code: *what exactly should the model predict?* There are several possible objectives, and they are **not** the same task.

| Objective | Type | Answers | Good for |
|---|---|---|---|
| Shortage before delivery (yes / no) | binary classification | *How likely is it that this station runs out before a box sent now arrives?* | a calibrated **risk %** per station |
| Risk class (low / medium / high / critical) | multiclass classification | the same, in buckets | display only: the buckets are arbitrary cut-offs of the probability |
| Time until empty | regression | *When* will it run out? | planning; but "when" is not "in danger": a station that empties in 10 min is safe if the car arrives in 2 |
| Which request first | ranking (pairwise classification) | *What should the car do next?* | the actual **decision** |

Three different things:
1. **Probability of a shortage:** a property of *one* station (Model B).
2. **When the shortage happens:** a time, also per station. We keep it as a column (`empty_in_s`) to explain, but do not train on it.
3. **Which station to visit:** a decision about *all* stations together, because serving one delays the others (Model A).

**Why not just "send the car to the highest risk"?** The brief's example: station X has risk 90 %, runs out in 20 min and is 18 min away; station Y has risk 85 %, runs out in 15 min and is 3 min away. Suppose the drive between X and Y also takes about 18 min.
- **X first (highest risk):** X is served at 18 min (in time), then Y at about 36 min: Y stood still for about **21 min**.
- **Y first:** Y is served at 3 min (in time), then X at about 21 min: X stood still for about **1 min**.

The lower-risk station should go first. The best order depends on the trips *between* stations, which a per-station probability cannot see. So Carkaroo uses:

> **risk model -> shows the danger and why;  ranker (checked by the exact optimizer) -> chooses the order;  car -> receives the destination.**

The rest of the notebook trains the ranker first (Part A), then the risk model (Part B).

# Part A: the ranker (which delivery goes first)

### A1. The factory and the teacher

Each situation has the car somewhere on the track and 2-6 open requests (a station needs a material). For every request we know: stock left, usage rate, time until empty, safety stock, how critical the station is, how long it has waited, and the car's trip time (from the track map).

The **teacher** (`agv_brain/optimizer.py`) tries *every* possible delivery order and computes its cost: seconds a station is stopped x criticality, time below safety stock, waiting and driving. That gives the exact best first delivery, but trying all orders grows as n! (7 requests = 5,040 orders), too slow to run for every decision on a bigger factory. The ranker learns to copy the teacher in a few milliseconds.

**Pairwise labels:** for each pair of requests (i, j) the label is 1 if starting with i gives a lower best cost than starting with j. The model then scores each request by how many "duels" it wins. Because inputs are times and numbers (never station names), the same model works on any track.

In [ ]:
from agv_brain.simulator import load_context
ctx = load_context()                         # our table-top track (config/)
s = ctx.settings
rows = [{"station": sid, "name": st.name, "criticality": st.criticality, "material": m,
         "stock": sm.stock, "use per min": sm.rate_per_min, "safety stock": sm.safety_stock, "capacity": sm.capacity}
        for sid, st in s.stations.items() for m, sm in st.materials.items()]
print(f"Track: {len(ctx.fmap.nodes)} nodes, car speed {s.car.speed_cm_s} cm/s")
pd.DataFrame(rows)

### A2. Generate the ranker dataset

**What:** random situations on our track (40 %) and on random factory layouts (60 %), every one labelled by the teacher. Random layouts make the model work on a track it has never seen ("domain randomization"); the car speed is also randomized so it keeps working after the real car is calibrated.

**Expected output:** 3,000 / 12,000 / 30,000 situations (quick / standard / full), with labels close to 50 % / 50 % (every pair appears in both directions).

In [ ]:
from agv_brain.dataset import generate, generate_plants
from agv_brain.features import PAIR_FEATURES, JOB_FEATURES

N_SCEN, N_LAYOUTS = SIZE["scen"], SIZE["layouts"]
t0 = time.time()
ds = generate(ctx, N_SCEN, seed=1, random_layouts=N_LAYOUTS, layout_share=0.6)
print(f"{N_SCEN} situations -> {len(ds.y)} pair rows x {ds.X.shape[1]} features in {time.time() - t0:.0f} s")
if SIZE["plants"]:                           # large factories at real-factory scale (2-340 m)
    t0 = time.time()
    plants = generate_plants(SIZE["plants"], s, seed=2)
    print(f"+ {len(plants.y)} pair rows from {SIZE['plants']} situations in large factories in {time.time() - t0:.0f} s")

requests = pd.DataFrame(ds.job_rows)         # one readable row per request
Path("data/generated").mkdir(parents=True, exist_ok=True)
np.savez_compressed("data/generated/pairs.npz", X=ds.X, y=ds.y.astype(int), w=ds.w, groups=ds.groups,
                    features=np.array(PAIR_FEATURES), layouts=np.array("notebook"))
requests.head()

### A3. Inspect the dataset

**What:** size, missing values, label balance, and the value ranges of the request features.

**How to read it:** there should be no missing values, the label mean should be about 0.5, and the times should look sensible (seconds; capped at 3600).

In [ ]:
print("requests:", requests.shape, "| pair rows:", ds.X.shape)
print("missing values in pair rows:", int(np.isnan(ds.X).sum()))
print("label balance (share of 1s):", round(ds.y.mean(), 3))
print("situations by layout type:", (requests.groupby("scenario").layout.first() == "ours").map({True: "our track", False: "random layout"}).value_counts().to_dict())
requests[JOB_FEATURES].describe().round(1).T

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
requests.stockout_s.clip(upper=1000).hist(bins=40, ax=ax[0]); ax[0].set_title("Time until empty (s)")
requests.trip_s.hist(bins=40, ax=ax[1]); ax[1].set_title("Car trip time (s)")
bins = pd.cut(requests.slack_s, [-3600, -60, 0, 60, 180, 600, 3600])
requests.groupby(bins, observed=True).teacher_best_first.mean().plot.bar(ax=ax[2])
ax[2].set_title("Share chosen first by the teacher, by slack"); ax[2].set_xlabel("slack = time to empty - trip (s)")
plt.tight_layout(); plt.show()

The right chart is the key idea: requests with **little or negative slack** (they would run out even if served right away) are chosen first much more often, but not always. Criticality, waiting time and where the other stations are also matter, and that is what the model has to learn.

### A4. Split and train

**Split by situation, never by row.** The pairs of one situation share most of their numbers. If some pairs of a situation were in training and others in the test, the test would be too easy (data leakage). So whole situations go to either training (80 %) or validation (20 %).

**Model:** `HistGradientBoostingClassifier`, many small decision trees, each correcting the previous ones. It handles numbers on very different scales without scaling, learns interactions ("little slack *and* high criticality"), trains in seconds and can be explained with feature importance.

In [ ]:
from agv_brain.ranker import PairwiseRanker

rng = np.random.default_rng(0)
scen = np.unique(ds.groups)
val_scen = rng.choice(scen, size=int(0.2 * len(scen)), replace=False)
val = np.isin(ds.groups, val_scen)
X_tr, y_tr, w_tr = ds.X[~val], ds.y[~val], ds.w[~val]
if SIZE["plants"]:
    X_tr, y_tr, w_tr = np.vstack([X_tr, plants.X]), np.concatenate([y_tr, plants.y]), np.concatenate([w_tr, plants.w])

t0 = time.time()
ranker = PairwiseRanker(max_iter=800).fit(X_tr, y_tr, w_tr)
pair_acc = ((ranker.predict_proba(ds.X[val]) >= 0.5) == ds.y[val]).mean()
print(f"trained in {time.time() - t0:.0f} s, {ranker.model.n_iter_} trees")
print(f"pairwise accuracy on held-out situations: {pair_acc:.3f}")

### A5. Evaluate: does it pick the same first delivery as the teacher?

**The capstone metric is top-1 accuracy:** in a brand-new situation, is the delivery the model sends the car to first the best one (the same as the exact optimizer)? Target: **at least 80 %**.

We test on new random situations (other seeds) on our track and on **factory layouts the model never saw**, with up to 7 requests (training stopped at 6). We also compare with simple rules: first come first served, nearest first, earliest stockout and hand-weighted rules. *Line stop* = seconds stations stand still waiting for material.

In [ ]:
from evaluate import run, summarize
from agv_brain.dataset import speed_variants
from agv_brain.layouts import layout_pool

N_TEST = SIZE["test"]
variants = speed_variants(ctx)
df_ours, _, _ = run(lambda r: variants[int(r.integers(len(variants)))], ranker, N_TEST, seed=20261103, n_range=(2, 7))
new_pool = layout_pool(100, 4242, s)
df_new, _, _ = run(lambda r: new_pool[int(r.integers(len(new_pool)))], ranker, N_TEST, seed=777, n_range=(2, 7))

table = pd.DataFrame({"our track: top-1": summarize(df_ours).top1_accuracy,
                      "unseen layouts: top-1": summarize(df_new).top1_accuracy,
                      "line stop (s)": summarize(df_ours).mean_line_stop_s,
                      "decision ms": summarize(df_ours).decision_ms_mean})
print(table.round(3).to_string())
print("Accuracy by number of requests (our track):")
print(df_ours[df_ours.policy == "ml"].groupby("n").top1.mean().round(3).to_dict())

**How to read it:** the ML ranker should be far above every simple rule and close to the optimizer's line-stop time. Accuracy drops a little with more requests (more ways to be wrong, and 7 is beyond training). The full-size model in the repository scores 91.6 % on our track and 91.1 % on unseen layouts (`reports/evaluation.md`); "quick" and "standard" use less data, so expect up to a few points less.

### A6. Why does it decide that? (feature importance)

**Permutation importance:** shuffle one input across all rows and measure how much the accuracy drops. A big drop means the model relies on that input. Each request feature appears three times in a pair (for i, for j, and i - j); they are added up here.

In [ ]:
from evaluate import grouped_importance
imp = grouped_importance(ranker, ds.X[val], ds.y[val], max_rows=8000 if MODE == "full" else 4000).sort_values()
imp.tail(10).plot.barh(figsize=(7, 4), title="What the ranker looks at most"); plt.xlabel("drop in pairwise accuracy"); plt.show()

### A7. (Optional) Learning from live operation (DAgger)

Random test situations are not exactly what the car meets when it runs the factory itself. DAgger fixes that: let the model run a simulated factory, label every situation it meets with the teacher, retrain on everything, repeat. In the repository this raised live accuracy from 76 % to 84 %. It is slow, so it is off by default.

In [ ]:
RUN_DAGGER = False
if RUN_DAGGER:
    ranker.save(OUT / "ranker_before_dagger.joblib")
    subprocess.run([sys.executable, "scripts/train_live.py", "--rounds", "1", "--episodes", "60",
                    "--model", str(OUT / "ranker_before_dagger.joblib"), "--out", str(OUT / "ranker.joblib")], check=True)
    ranker = PairwiseRanker.load(OUT / "ranker.joblib")
ranker.save(OUT / "ranker.joblib")
print("saved", OUT / "ranker.joblib")

# Part B: the shortage-risk model (will this station run out?)

### B1. Defining the target, carefully

For one station and one material, at a moment *t*:

> **`shortage_before_delivery` = 1** if the station's *real* usage between *t* and *t + ETA* is at least the stock it has at *t*.
> ETA = the car's trip to bring a box if it is sent **now** + the trips already queued ahead of it.

So the model answers: *if we sent a box right now, would the station still run dry before it arrives?* The risk is the **probability** of that event, P(shortage before delivery | what we know now).

**Why this is not a hidden formula.** If usage were constant, the answer would be `stock / rate < ETA` and no ML would be needed. In our simulation usage changes **after** *t*: the shift changes, new production orders start, unannounced surges hit, maintenance stops the line, and usage is noisy. The label is computed from that **future**; the inputs only see the **past** readings and the **production plan**. We keep `stock / rate < ETA` as the baseline the model must beat.

**Avoiding leakage:** every input is built from readings up to *t* and the plan as known at *t* (orders count only once announced). The future is used **only** for the label. `empty_in_s` (when it would run out) is kept for explanation but is **never** an input.

### B2. The factory simulation: changing demand

`agv_brain/demand.py` simulates one station-material over an hour, every 2 s:

`rate(t) = base rate x shift multiplier x (1 + running orders) x surge x (0 during downtime) x noise`

`stock(t) = stock(t - 2 s) + deliveries - used` (never below 0); the load cell reports whole units.

A simple replenishment loop orders a box when the reading reaches a reorder point; the box arrives after 1-4 car trips. Time is compressed for the table-top factory: a "shift" lasts minutes, not hours. Let's look at one station:

In [ ]:
from agv_brain.demand import random_plan, simulate_station
r = np.random.default_rng(3)
tr = simulate_station(r, random_plan(r), station="S1", material="A", criticality=3, capacity=60, safety_stock=6,
                      base_rate_per_min=3.0, delivery_qty=20, trip_s=60)
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(tr.t / 60, tr.rate * 60, lw=0.7); ax[0].set_ylabel("usage (units/min)")
for w in tr.surges: ax[0].axvspan(w.start_s / 60, w.end_s / 60, color="red", alpha=0.15)
for w in tr.downtime: ax[0].axvspan(w.start_s / 60, w.end_s / 60, color="grey", alpha=0.3)
ax[0].set_title("Usage: shifts (steps), orders (bumps), surges (red), downtime (grey), noise")
ax[1].plot(tr.t / 60, tr.reading, lw=1); ax[1].axhline(tr.safety_stock, color="orange", ls="--", label="safety stock")
ax[1].set_ylabel("stock (load cell)"); ax[1].set_xlabel("minutes"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"{len(tr.orders)} production orders, {len(tr.surges)} surges, {len(tr.deliveries)} deliveries")

### B3. Generate the risk dataset

**One row = one station-material at one moment.** Each timeline is one simulated hour of one factory (our track or one of 200 random layouts). After a 15-minute warm-up (so the 15-minute usage window has data), a row is taken about every 45 s. A separate set of timelines in factories built from another seed is kept as the **unseen-layouts test**.

**Expected output:** about 14,000 / 29,000 / 43,000 rows (quick / standard / full), with roughly 25-30 % shortages, no missing values and no duplicates.

In [ ]:
from agv_brain.risk_dataset import (LABEL, RISK_FEATURES, generate_risk, training_layouts, unseen_layouts,
                                    split_by_timeline, naive_rule, plan_rule)
N_TL, N_UNSEEN = SIZE["risk_tl"], SIZE["unseen"]
t0 = time.time()
risk = generate_risk(training_layouts(ctx), N_TL, seed=11)
risk_unseen = generate_risk(unseen_layouts(ctx), N_UNSEEN, seed=1011)
print(f"{len(risk)} rows from {N_TL} timelines (+{len(risk_unseen)} unseen-layout rows) in {time.time() - t0:.0f} s")
risk.head()

### B4. Inspect the dataset (quality checks)

In [ ]:
print("shape:", risk.shape)
print("missing values:", int(risk[RISK_FEATURES].isna().sum().sum()))
print("duplicate rows:", int(risk.duplicated(subset=RISK_FEATURES).sum()))
print("negative stock / ETA:", int((risk.stock < 0).sum()), int((risk.eta_s < 0).sum()))
print("shortage rate:", round(risk[LABEL].mean(), 3))
print(risk.dtypes.value_counts().to_dict())
risk[RISK_FEATURES].describe().round(2).T

### B5. Exploratory data analysis

What to look for: realistic ranges, shortages more frequent in busy shifts and with long ETAs, and a clear but **not perfect** separation between shortages and safe moments (if it were perfect, we would not need a model).

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
risk.stock.hist(bins=40, ax=ax[0, 0]); ax[0, 0].set_title("Stock (units)")
risk.rate_est_per_min.clip(upper=20).hist(bins=40, ax=ax[0, 1]); ax[0, 1].set_title("Measured usage (units/min)")
risk.groupby("shift")[LABEL].mean().reindex(["day", "evening", "night"]).plot.bar(ax=ax[0, 2])
ax[0, 2].set_title("Shortage rate by shift")
smp = risk.sample(min(4000, len(risk)), random_state=0)
ax[1, 0].scatter(smp.stock, smp.rate_est_per_min.clip(upper=20), c=smp[LABEL], cmap="coolwarm", s=4, alpha=0.6)
ax[1, 0].set_xlabel("stock"); ax[1, 0].set_ylabel("usage/min"); ax[1, 0].set_title("Usage vs stock (red = shortage)")
risk.groupby(pd.cut(risk.eta_s, [0, 60, 120, 180, 300, 900]), observed=True)[LABEL].mean().plot.bar(ax=ax[1, 1])
ax[1, 1].set_title("Shortage rate by time until a box arrives (s)")
risk.groupby(pd.cut(risk.margin_naive_s, [-3600, -120, -30, 0, 30, 120, 3600]), observed=True)[LABEL].mean().plot.bar(ax=ax[1, 2])
ax[1, 2].set_title("Shortage rate by (stock/rate - ETA) (s)")
plt.tight_layout(); plt.show()

The bottom-right chart shows why a formula is not enough: right around 0 (the formula's dividing line) the outcome is uncertain, because the future usage is not the past usage.

### B6. Preprocessing

- **Missing values:** none (checked above), so no filling is needed. In the live system, a station without enough readings uses its planned rate (the forecaster's fallback).
- **Categorical variables:** the shift *name* is not an input; the shift's planned **workload multiplier** is (a number), which carries the same information and also works for factories with other shift patterns.
- **Scaling:** tree models do not need it. Logistic regression does, so it gets a `StandardScaler` inside a pipeline (fitted on training rows only).
- **Removed columns:** `timeline`, `t_s`, `layout`, `station`, `material` (identity: the model must not learn "station S1 is always urgent"), `empty_in_s` and the label (they are the future).
- **Calculated features:** `tte_naive_s = stock / rate`, `margin_naive_s = tte - ETA` and `planned_margin = stock - planned usage until arrival` are given to the model next to the raw values they come from. They make the pattern easier to learn; section B9 tests the model without them.

### B7. Split by timeline: 70 % train, 15 % validation, 15 % test

Rows from one timeline, a minute apart, are almost copies. A random row split would put near-copies in training and test and make the score look better than it is. So **whole timelines** go to one set only. The validation set is used to tune and calibrate; the test set is used **once**, at the end.

In [ ]:
tr_m, va_m, te_m = split_by_timeline(risk, seed=0)
assert not (set(risk[tr_m].timeline) & set(risk[te_m].timeline)), "a timeline is in two sets!"
X, y = risk[RISK_FEATURES], risk[LABEL].to_numpy()
for name, m in [("train", tr_m), ("validation", va_m), ("test", te_m)]:
    print(f"{name:10} {m.sum():6} rows, {risk[m].timeline.nunique():3} timelines, shortage rate {y[m].mean():.3f}")

### B8. Train and compare models

- **Logistic regression:** a weighted sum of the inputs through an S-curve. Fast and readable, but only straight-line boundaries.
- **Random forest:** hundreds of deep trees on random parts of the data, averaged. Strong, but its probabilities are often too cautious.
- **Gradient boosting (HistGradientBoosting):** trees built one after another, each fixing the previous errors. Usually the most accurate on tables of numbers, fast, and works with SHAP. (XGBoost and LightGBM are the same idea in other libraries; scikit-learn's version avoids an extra dependency on the laptop.)

Plus the two formulas as baselines.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss, confusion_matrix)
from agv_brain.risk_model import RiskModel, make_hgb, FEATURE_LABELS

def scores(y_true, p, pred):
    return dict(accuracy=accuracy_score(y_true, pred), precision=precision_score(y_true, pred, zero_division=0),
                recall=recall_score(y_true, pred), f1=f1_score(y_true, pred), roc_auc=roc_auc_score(y_true, p),
                pr_auc=average_precision_score(y_true, p), brier=brier_score_loss(y_true, p))

yt, test = y[te_m], risk[te_m]
results = {"Plain formula (stock/rate < ETA)": scores(yt, naive_rule(test), naive_rule(test)),
           "Plan formula (planned use >= stock)": scores(yt, plan_rule(test), plan_rule(test))}
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X[tr_m], y[tr_m])
rf = RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(X[tr_m], y[tr_m])
hgb = make_hgb().fit(X[tr_m], y[tr_m])
for name, m in [("Logistic regression", lr), ("Random forest", rf), ("Gradient boosting", hgb)]:
    p = m.predict_proba(X[te_m])[:, 1]
    results[name] = scores(yt, p, p >= 0.5)
pd.DataFrame(results).T.round(4)

### B9. Evaluation: why accuracy alone can mislead

About 3 out of 4 moments are safe. A "model" that always answers *safe* would get about 73 % accuracy and catch **zero** shortages. So we also look at:
- **Recall:** of the real shortages, how many did we catch? (a missed shortage = a stopped line)
- **Precision:** of the alarms, how many were real? (a false alarm = an unnecessary trip)
- **ROC-AUC / PR-AUC:** how well the *probability* separates the two classes, at every threshold.
- **Brier score:** the mean squared error of the probability (lower is better): it checks the probability itself, not just the yes/no.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
p_hgb = hgb.predict_proba(X[te_m])[:, 1]
print("Always 'safe':", round(accuracy_score(yt, np.zeros_like(yt)), 3), "accuracy, 0 shortages caught")
print("Confusion matrix (gradient boosting, risk >= 0.5):\n", confusion_matrix(yt, p_hgb >= 0.5))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(yt, p_hgb, ax=ax[0], name="gradient boosting")
PrecisionRecallDisplay.from_predictions(yt, p_hgb, ax=ax[1], name="gradient boosting")
plt.tight_layout(); plt.show()

### B10. Improving the model

1. **Hyperparameters**, tuned on the **validation** timelines (never the test set).
2. **Class imbalance:** compare with `class_weight="balanced"` (shortages count more in training).
3. **Feature sets:** what does each group of inputs add?
4. **Calibration:** make "risk 0.8" really mean "80 % of such moments ran out", then choose the alarm threshold that catches 95 % of shortages.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
grid = [(lr_, leaves) for lr_ in (0.03, 0.06, 0.1) for leaves in (15, 31)] if MODE == "full" else [(0.06, 15), (0.06, 31), (0.1, 31)]
tuning = []
for lr_, leaves in grid:
    m = HistGradientBoostingClassifier(learning_rate=lr_, max_leaf_nodes=leaves, max_iter=400, min_samples_leaf=40,
                                       early_stopping=True, random_state=0).fit(X[tr_m], y[tr_m])
    pv = m.predict_proba(X[va_m])[:, 1]
    tuning.append(dict(learning_rate=lr_, leaves=leaves, val_auc=roc_auc_score(y[va_m], pv), val_acc=accuracy_score(y[va_m], pv >= 0.5)))
print(pd.DataFrame(tuning).round(4))

bal = HistGradientBoostingClassifier(max_iter=400, class_weight="balanced", random_state=0).fit(X[tr_m], y[tr_m])
pb = bal.predict_proba(X[te_m])[:, 1]
print(f"\nclass_weight='balanced': recall {recall_score(yt, pb >= 0.5):.3f} (vs {recall_score(yt, p_hgb >= 0.5):.3f}), "
      f"precision {precision_score(yt, pb >= 0.5):.3f} (vs {precision_score(yt, p_hgb >= 0.5):.3f})")

PLAN = ["shift_mult", "next_shift_mult", "to_shift_change_s", "orders_active", "orders_boost", "orders_coming_boost",
        "downtime_now", "downtime_in_eta_s", "planned_use_eta", "planned_margin"]
sets = {"all inputs": RISK_FEATURES,
        "raw inputs only (no planned_use_eta / planned_margin)": [f for f in RISK_FEATURES if f not in ("planned_use_eta", "planned_margin")],
        "without the production plan": [f for f in RISK_FEATURES if f not in PLAN],
        "only stock, rate, ETA": ["stock", "rate_est_per_min", "eta_s"]}
for name, cols in sets.items():
    m = make_hgb().fit(X.loc[tr_m, cols], y[tr_m]); q = m.predict_proba(X.loc[te_m, cols])[:, 1]
    print(f"{name:55} accuracy {accuracy_score(yt, q >= 0.5):.3f}  ROC-AUC {roc_auc_score(yt, q):.3f}")

**How to read the feature sets:** removing the production plan costs several points: knowing the shift change, the orders and the downtime ahead is what lets the model look into the future. With only stock, rate and ETA, the model is about as good as the plain formula, which is what we should expect: it then has no information about the future that the formula lacks.

In [ ]:
from sklearn.calibration import calibration_curve
risk_model = RiskModel.train(risk[tr_m], y[tr_m], risk[va_m], y[va_m])   # trees + isotonic calibration on validation
p_risk = risk_model.predict_proba(test)
print(f"alarm threshold (95 % recall on validation): {risk_model.threshold:.2f}")
print(pd.DataFrame({"risk >= 0.5": scores(yt, p_risk, p_risk >= 0.5),
                    f"alarm >= {risk_model.threshold:.2f}": scores(yt, p_risk, risk_model.predict(test))}).T.round(4))
frac, mean_p = calibration_curve(yt, p_risk, n_bins=10, strategy="quantile")
fr2, mp2 = calibration_curve(yt, p_hgb, n_bins=10, strategy="quantile")
plt.figure(figsize=(4.5, 4.2)); plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.plot(mp2, fr2, "o-", alpha=0.5, label="before calibration"); plt.plot(mean_p, frac, "o-", label="calibrated")
plt.xlabel("predicted risk"); plt.ylabel("share that really ran out"); plt.legend(); plt.title("Calibration"); plt.show()

### B11. Testing on factories it never saw, and on close calls

In [ ]:
yu = risk_unseen[LABEL].to_numpy(); pu = risk_model.predict_proba(risk_unseen)
print(f"Unseen layouts: risk model {accuracy_score(yu, pu >= 0.5):.3f} | plan formula {accuracy_score(yu, plan_rule(risk_unseen)):.3f} | "
      f"plain formula {accuracy_score(yu, naive_rule(risk_unseen)):.3f}")
close = (test.margin_naive_s.abs() < 120).to_numpy()
print(f"Close calls ({close.sum()} rows, formula within 2 min of the line): risk model {accuracy_score(yt[close], p_risk[close] >= 0.5):.3f} | "
      f"plan formula {accuracy_score(yt[close], plan_rule(test[close])):.3f} | plain formula {accuracy_score(yt[close], naive_rule(test[close])):.3f}")

**Honest reading.** Most moments are clearly safe or clearly not, so even the formulas score high overall. The model's real gains are on the **close calls** and in giving a **calibrated probability** instead of a yes/no. Compare your numbers with `reports/risk_model.md` (full-size data).

### B12. Explainability

**Permutation importance:** shuffle one input and see how much the ROC-AUC drops. Inputs that carry the same information share the credit, so look at it together with the feature-set table above.

**SHAP (optional):** for one prediction, how much each input pushed the risk up or down from the average. Set `USE_SHAP = True` (it installs the `shap` library in Colab only; the laptop does not need it).

In [ ]:
from sklearn.inspection import permutation_importance
smp = test.sample(min(5000, len(test)), random_state=0)
pi = permutation_importance(risk_model.model, smp[RISK_FEATURES], smp[LABEL], scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=[FEATURE_LABELS[f] for f in RISK_FEATURES]).sort_values()
imp.tail(10).plot.barh(figsize=(7, 4), title="What the risk model looks at most"); plt.xlabel("drop in ROC-AUC"); plt.show()

USE_SHAP = False
if USE_SHAP:
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap
    explainer = shap.TreeExplainer(hgb)
    sv = explainer(smp[RISK_FEATURES].head(500))
    shap.plots.beeswarm(sv, max_display=12)

### B13. Save the model

In [ ]:
risk_model.save(OUT / "risk_model.joblib")
print("saved", OUT / "risk_model.joblib", "| sklearn", risk_model.meta["sklearn"], "| threshold", round(risk_model.threshold, 3))

# Part C: both models together

### C1. Try your own stations

Enter the current situation of a few stations on our track. You get each station's **risk** (Model B), the **delivery order** the ranker chooses (Model A), and the **reasons**, including "what if another station went first", checked with the exact optimizer.

Rates are in units per **minute** (the table-top factory runs in compressed time). The defaults:
- **S1 / A** ("X" in the brief): little stock, used fast, a critical line.
- **S3 / B** ("Y"): more stock, used slowly: safe.
- **S2 / C**: only 2 units, but `stock / rate` says 80 s, longer than the 58 s trip, so the plain formula calls it safe. A production order is running (+80 % usage), and the model sees that.

Change the numbers and run the cell again.

In [ ]:
from agv_brain.models import CarState, Job
from agv_brain.simulator import job_leg, simulate
from agv_brain.baselines import rollout
from agv_brain.explain import explain
from agv_brain.risk_dataset import manual_features

def predict_stations(stations, car_node=None, shift_mult=1.0, next_shift_mult=1.0, to_shift_change_s=3600):
    car = CarState(car_node or s.car.dock, s.car.start_heading)
    jobs, feats = [], []
    for st in stations:
        sid, m = st["station"], st["material"]
        conf = s.stations[sid]
        job = Job(f"{m}@{sid}", m, sid, s.materials[m].rack, st["stock"], st["rate_per_min"] / 60,
                  conf.materials[m].safety_stock, conf.criticality, st.get("wait_s", 0.0))
        trip = job_leg(ctx, car, job).duration_s
        jobs.append(job)
        feats.append(manual_features(stock=st["stock"], rate_per_min=st["rate_per_min"], trip_s=trip,
                                     capacity=conf.materials[m].capacity, safety_stock=conf.materials[m].safety_stock,
                                     criticality=conf.criticality, shift_mult=shift_mult, next_shift_mult=next_shift_mult,
                                     to_shift_change_s=to_shift_change_s, orders_active=st.get("orders", 0),
                                     orders_boost=st.get("orders_boost", 0.0)))
    risk_p = risk_model.predict_proba(feats)
    order = rollout(ranker, ctx, jobs, car)
    plan = simulate(ctx, order, car)
    table = pd.DataFrame([{"request": j.id, "stock": j.stock, "use/min": round(j.rate_per_s * 60, 1),
                           "empty in (s)": round(j.stockout_s), "trip (s)": round(f["trip_s"]),
                           "risk": f"{p:.0%}", "order": [o.id for o in order].index(j.id) + 1}
                          for j, f, p in zip(jobs, feats, risk_p)]).sort_values("order")
    print(table.to_string(index=False))
    print(f"\n-> Send the car to: {order[0].station} with material {order[0].material}\n")
    for line in explain(ctx, jobs, car, plan):
        print(" -", line)
    for j, f in zip(jobs, feats):
        top = risk_model.explain(f, risk[tr_m], top=2)
        print(f"   risk of {j.id}: " + ", ".join(f"{FEATURE_LABELS[k]} ({v:+.0%})" for k, v in top))
    return table

predict_stations([
    {"station": "S1", "material": "A", "stock": 4,  "rate_per_min": 5.0},   # "X": used fast
    {"station": "S3", "material": "B", "stock": 12, "rate_per_min": 1.0},   # "Y": used slowly
    {"station": "S2", "material": "C", "stock": 2,  "rate_per_min": 1.5, "orders": 1, "orders_boost": 0.8},
])

# Part D: export to the laptop

**Joblib, not pickle or ONNX:**
- `joblib` is scikit-learn's recommended way to save models (pickle underneath, but efficient with the large number arrays inside trees).
- **ONNX** would be useful if a microcontroller had to run the model. It does not: the STM32 / BW20 on the car only needs a **destination ID** (a few bytes over MQTT). The models run on the laptop, which has Python. Tree models with 800 trees would not fit a small microcontroller's memory anyway.
- Only load `.joblib` files you made yourself: loading runs Python code.

**Steps:** the models are saved to Google Drive (`MyDrive/carkaroo_models/<date-time>/`) and downloaded as a zip. When you are next at the laptop that runs the factory, copy `ranker.joblib` and `risk_model.joblib` into `models/` (keep the old ones as a backup) and run `python scripts/evaluate.py` and `python scripts/evaluate_risk.py` to check them before the exhibition.

In [ ]:
SAVE_TO_DRIVE = True               # keep a copy in Google Drive (asks for permission the first time)
archive = shutil.make_archive(str(WORK / "carkaroo_models"), "zip", OUT)
print("models:", sorted(p.name for p in OUT.iterdir()), "->", archive)
if IN_COLAB:
    from google.colab import files
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/carkaroo_models") / time.strftime("%Y-%m-%d_%H-%M")
        shutil.copytree(OUT, dest)
        print("saved to Google Drive:", dest)
    files.download(archive)

## What comes next

1. **Website:** show each station's risk % and its top reasons next to the ranker's choice (the dashboard reads `models/risk_model.joblib`).
2. **Live inputs:** feed the risk model from the real load cells: the forecaster already gives the usage rate; the shift plan and production orders come from the Setup page.
3. **Cloud:** the decision and the risks are already published over MQTT; the HiveMQ cloud mirror makes them visible remotely.